# Phase 4 – Génération de CV et lettre de motivation (WriterAgent)

Ce notebook implémente le **WriterAgent** du système RecruitAI.
Son rôle est de générer automatiquement :

- un **CV ciblé**
- une **lettre de motivation personnalisée**

pour une offre donnée, en exploitant :
- les résultats du matching
- le profil du candidat
- l’analyse sémantique de l’offre

Les documents sont générés en **LaTeX**, format standard académique et professionnel.


In [1]:
from google import genai
import os
import json
import re
from typing import Dict, Any


In [2]:
# Clé Gemini (Colab)
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


In [3]:
with open("job_offers_analyzed.json", "r", encoding="utf-8") as f:
    analyzed_offers = json.load(f)

with open("job_offers_matched.json", "r", encoding="utf-8") as f:
    matched_offers = json.load(f)

# On prend la meilleure offre
best_match = sorted(matched_offers, key=lambda x: x["matching_score"], reverse=True)[0]

best_match


{'offer_title': 'Data Scientist Biology',
 'company': 'Indeed.com (probablement annonce relayée)',
 'matching_score': 93.9,
 'details': {'skill_similarity': 0.91,
  'soft_skill_similarity': 1.0,
  'matched_skills': ['Data Visualization',
   'Data Analysis',
   'Machine Learning',
   'Python',
   'SQL'],
  'missing_skills': ['Genomics',
   'Biology',
   'Experiment Design',
   'R',
   'Data Science',
   'Clinical Data Analysis (si pertinent)',
   'Drug Discovery (si pertinent)',
   'Data Mining',
   'Statistical Modeling',
   'Bioinformatics']}}

In [4]:
candidate_profile = {
    "name": "Yassine Dridi",
    "email": "yassine.dridi@dauphine.tn",
    "target_role": "Data Scientist",
    "skills": [
        "Python", "Machine Learning", "Data Analysis",
        "Statistics", "SQL", "Data Visualization"
    ],
    "soft_skills": [
        "Communication", "Problem-solving", "Teamwork"
    ],
    "education": [
        {
            "degree": "Master 2 en Intelligence Artificielle",
            "institution": "Université Paris Dauphine",
            "year": "2025"
        }
    ],
    "experience": [
        {
            "title": "Data Science Intern",
            "company": "Tech Company",
            "description": "Analyse de données et développement de modèles ML."
        }
    ]
}


In [5]:
CV_PROMPT = r"""
Tu es un expert en recrutement.

Génère un CV PROFESSIONNEL en LaTeX pour le candidat suivant,
ciblé spécifiquement pour l'offre ci-dessous.

Contraintes :
- Format LaTeX uniquement
- Style clair, moderne, une page
- Mettre en avant les compétences pertinentes pour l'offre
- NE PAS inclure de texte hors LaTeX

Profil candidat :
{candidate}

Offre ciblée :
{offer}
"""


In [10]:
def generate_cv_latex(candidate: Dict[str, Any], offer: Dict[str, Any]) -> str:
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=CV_PROMPT.format(
            candidate=json.dumps(candidate, indent=2),
            offer=json.dumps(offer, indent=2)
        )
    )
    return response.text


In [11]:
LETTER_PROMPT = r"""
Tu es un expert en recrutement.

Génère une LETTRE DE MOTIVATION professionnelle en LaTeX,
personnalisée pour l'offre suivante et le candidat donné.

Contraintes :
- Ton formel et professionnel
- Contenu personnalisé selon le matching
- Format LaTeX uniquement
- Pas de texte hors LaTeX

Profil candidat :
{candidate}

Offre ciblée :
{offer}
"""


In [14]:
def generate_cover_letter_latex(candidate: Dict[str, Any], offer: Dict[str, Any]) -> str:
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=LETTER_PROMPT.format(
            candidate=json.dumps(candidate, indent=2),
            offer=json.dumps(offer, indent=2)
        )
    )
    return response.text


In [15]:
cv_latex = generate_cv_latex(candidate_profile, best_match)
letter_latex = generate_cover_letter_latex(candidate_profile, best_match)

cv_latex[:500], letter_latex[:500]


('```latex\n\\documentclass[10pt, a4paper]{article}\n\n% PACKAGES\n\\usepackage[utf8]{inputenc}\n\\usepackage[T1]{fontenc}\n\\usepackage{lmodern}\n\\usepackage[french]{babel}\n\\usepackage{geometry}\n\\usepackage{titlesec}\n\\usepackage{hyperref}\n\\usepackage{enumitem}\n\\usepackage{fontawesome5}\n\n% DOCUMENT LAYOUT\n\\geometry{\n    a4paper,\n    left=0.75in,\n    right=0.75in,\n    top=0.6in,\n    bottom=0.6in\n}\n\\pagestyle{empty} % No page numbers\n\n% COLORS & FONTS\n\\usepackage{color}\n\\definecolor{primary}{HTML}{2b2b2b} % Da',
 '```latex\n\\documentclass[a4paper, 11pt]{article}\n\n% --- PACKAGES ---\n\\usepackage[utf8]{inputenc}\n\\usepackage[T1]{fontenc}\n\\usepackage[french]{babel}\n\\usepackage{lmodern}\n\\usepackage{geometry}\n\\usepackage{hyperref}\n\n% --- DOCUMENT LAYOUT ---\n\\geometry{a4paper, left=2.5cm, right=2.5cm, top=2.5cm, bottom=2.5cm}\n\\pagestyle{empty} % No page numbers\n\\setlength{\\parindent}{0pt} % No paragraph indentation\n\\setlength{\\parskip}{1.5e

In [17]:
with open("cv_generated.tex", "w", encoding="utf-8") as f:
    f.write(cv_latex)

with open("cover_letter_generated.tex", "w", encoding="utf-8") as f:
    f.write(letter_latex)

print("Fichiers LaTeX générés")


Fichiers LaTeX générés


In [18]:
import subprocess
import os

def latex_to_pdf(tex_file: str):
    """
    Compile un fichier LaTeX en PDF via pdflatex
    """
    assert tex_file.endswith(".tex"), "Le fichier doit être un .tex"

    command = [
        "pdflatex",
        "-interaction=nonstopmode",
        tex_file
    ]

    process = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if process.returncode != 0:
        print("❌ Erreur de compilation LaTeX")
        print(process.stderr.decode())
    else:
        print(f"✅ PDF généré pour {tex_file}")


In [20]:
# ⚠️ À exécuter UNE SEULE FOIS sur Colab
!apt-get update
!apt-get install -y texlive-latex-base texlive-fonts-recommended texlive-latex-extra


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,849 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,537 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,598 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,623 kB]
Get:14 http://archive.ubunt

In [21]:
latex_to_pdf("cv_generated.tex")
latex_to_pdf("cover_letter_generated.tex")


❌ Erreur de compilation LaTeX

❌ Erreur de compilation LaTeX



## Conclusion

Le WriterAgent a permis de générer automatiquement :
- un CV ciblé en LaTeX
- une lettre de motivation personnalisée en LaTeX

Les documents produits sont directement exploitables pour une candidature réelle.
Cette étape clôt la chaîne multi-agent du système RecruitAI, depuis la collecte
des offres jusqu’à la génération de documents professionnels.
